# SignLink — Classical-ML Experiment (pooled data, your algorithm)

Run each cell top to bottom. Only **CELL 5** changes per person (the `ALGO`).

CELL 3 is preset for the team's Drive layout and auto-loads the AUTSL
base (the held-out test signers). If your folder differs, edit `SRC`.

## CELL 1 — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/SignLink'

## CELL 2 — clone/update the repo, install deps

In [ ]:
import os
!git config --global --add safe.directory /content/Sign-to-Text
if not os.path.isdir('/content/Sign-to-Text'):
    !git clone https://github.com/Catejsj/Sign-to-Text.git /content/Sign-to-Text
else:
    !cd /content/Sign-to-Text && git fetch && git reset --hard origin/main
%cd /content/Sign-to-Text/khmer_sign_recognizer
!pip install -q scikit-learn numpy

## CELL 3 — load the AUTSL base + pool everyone's recordings

Unzips the AUTSL base (test signers) and routes every recording by its
own metadata, forcing all into one language so the 3 people merge.

In [ ]:
# Point at the recordings folder (NOT all of MyDrive, or it grabs old Khmer data).
SRC = '/content/drive/MyDrive/SignLink/data/sequences_v2/turkish'
FORCE_LANG = 'autsl'   # merge everyone into one language

import shutil, json, zipfile
from pathlib import Path

DATA = Path('/content/Sign-to-Text/khmer_sign_recognizer/data/sequences_v2')
DATA.mkdir(parents=True, exist_ok=True)

# AUTSL base = real Turkish signers, carry the held-out val/test split.
base_zip = Path(SRC) / 'autsl_base.zip'
if base_zip.exists():
    with zipfile.ZipFile(base_zip) as z:
        z.extractall(DATA)
    print('unzipped AUTSL base ->', DATA / 'autsl')
else:
    print('WARNING: no autsl_base.zip at', base_zip, '- no test signers!')

# Pool everyone's recordings, routing each by its own json metadata.
copied = skipped = 0
by_signer = {}
for npy in Path(SRC).rglob('*.npy'):
    js = npy.with_suffix('.json')
    if not js.exists():
        skipped += 1; continue
    try:
        meta = json.loads(js.read_text(encoding='utf-8'))
    except Exception:
        skipped += 1; continue
    label = meta.get('label', npy.parent.name)
    lang = FORCE_LANG or meta.get('language', 'unknown')
    meta['language'] = lang
    dst = DATA / lang / label
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(npy, dst / npy.name)
    (dst / js.name).write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
    copied += 1
    sg = meta.get('signer_id', '?')
    by_signer[sg] = by_signer.get(sg, 0) + 1

print(f'pooled {copied} recordings (skipped {skipped})')
print('by signer:', by_signer)

## CELL 4 — (re)generate synthetic on the pooled real data

Makes synthetic for every signer consistently. Your uploaded synthetic
on Drive is untouched — this only affects the Colab training copy.

In [ ]:
LANG = 'autsl'
!python scripts/generate_synthetic.py --language {LANG} --per-take 1 --clean

## CELL 5 — >>> SET YOUR ALGORITHM <<< then run real vs real+synthetic

Pick ONE: `lda` | `logreg` | `rf` | `svm` | `nb` | `tree` | `knn` (lda was best for us)

In [ ]:
ALGO = 'lda'

print('=== RUN A: real only ===')
!python scripts/run_baseline.py --algo {ALGO} --lang {LANG} --mode real

print('\n=== RUN B: real + synthetic ===')
!python scripts/run_baseline.py --algo {ALGO} --lang {LANG} --mode both

## CELL 6 (optional) — save the results table back to Drive

In [ ]:
res = Path('/content/Sign-to-Text/khmer_sign_recognizer/data/experiments/baseline_results.csv')
if res.exists():
    out = Path(DRIVE) / 'experiments'
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(res, out / f'baseline_results_{ALGO}.csv')
    print('saved results to', out / f'baseline_results_{ALGO}.csv')
    print(res.read_text())